# AgentRegistry, end to end: scaffold a dice agent, add approved MCP tools, deploy to kagent **and** AWS Bedrock AgentCore, then govern the tools

A developer builds an agent with `arctl`, runs it locally, pulls in **approved MCP tool servers** from the registry catalog, publishes the agent, and deploys the *same* published agent to two runtimes (Solo Enterprise for kagent on kind, and AWS Bedrock AgentCore). Finally a platform owner restricts which MCP tools the agent may call with an **AccessPolicy**, enforced at an agentgateway waypoint.

> **Kernel:** pick **Bash** (top-right). One-time engineer setup lives in `scripts/` (see `README.md`).

## Open the consoles

Run this once **in a terminal** — it port-forwards the kagent UI and opens both tabs (no `/etc/hosts` edit needed):

| Console | URL | Login |
|---|---|---|
| AgentRegistry UI | http://localhost:12121 | — |
| kagent UI | http://localhost:18007 | alice / alice |
| AWS Bedrock AgentCore | the AWS console (manual) | — |

## Connect to the platform

Loads creds, puts `arctl` on the path, mints a catalog token.

In [ ]:
rm -rf agentdemo

In [ ]:
[ -d agentregistry-agentcore-kind ] && cd agentregistry-agentcore-kind || :; source scripts/connect.sh

## 1. Browse the approved tool-server catalog

Rather than wiring arbitrary MCP servers, developers pull from an **approved catalog**. Open the **AgentRegistry UI** (http://localhost:12121) → **Tool Servers** to see `everything-server` and `my-mcp`. The same list from the CLI:

In [ ]:
arctl get mcpservers

## 2. Create the agent, wired to the approved tools

Scaffold the agent and reference the approved servers straight from the catalog with `--mcp` (repeatable). arctl records them in `agent.yaml` under `spec.mcpServers`, resolved into tool servers beside the agent at deploy time. The generated agent also has two **local** tools, `roll_die` and `check_prime`.

In [ ]:
arctl init agent agentdemo --framework adk --language python --model-provider anthropic --model-name claude-haiku-4-5 --mcp demo/everything-server@latest --mcp demo/my-mcp@latest

In [ ]:
cat agentdemo/agentdemo/agent.py

In [ ]:
cat agentdemo/agent.yaml

## 3. Build and run it locally

Build the image:

In [ ]:
arctl build ./agentdemo

Then chat with the agent **in a terminal** — `arctl run` is interactive, so copy the command below and run it in your shell:

Ask it (paste into the chat):

It uses the local `roll_die` + `check_prime` tools. The approved catalog tools (`sum`, `echo`, `word_count`, ...) come online when the agent is **deployed**: the registry stands up the MCP servers beside it (you'll see them next).

## 4. Build (multi-arch), publish, and push the source

Multi-arch so the same image runs on kagent (arm64 here) and AgentCore (amd64).

In [ ]:
arctl build ./agentdemo --platform linux/amd64,linux/arm64 --push

In [ ]:
arctl apply -f agentdemo/agent.yaml

AgentCore clones the agent **source** from git at deploy time, so push it to your agent repo (the engineer set `AGENT_GIT_URL` in `.env.local`):

In [ ]:
# AgentCore clones the agent SOURCE from git at deploy time, so push agentdemo/
# to your agent repo ($AGENT_GIT_URL, from .env.local).
set -o pipefail
: "${AGENT_GIT_URL:?set AGENT_GIT_URL in .env.local}"
SLUG="${AGENT_GIT_URL#https://github.com/}"; SLUG="${SLUG%.git}"; BR="${AGENT_GIT_BRANCH:-main}"
PUSH_URL="https://x-access-token:$(gh auth token)@github.com/${SLUG}.git"
T="$(mktemp -d)"; cp -R agentdemo "$T/agentdemo"; rm -rf "$T/agentdemo/.git" "$T/agentdemo/.venv"
( cd "$T" \
  && git init -qb "$BR" && git add -A \
  && git -c user.email=demo@local -c user.name=demo commit -qm "agentdemo source" \
  && git remote add origin "$PUSH_URL" && git push -fq origin "$BR" )
rm -rf "$T"
echo "pushed agentdemo/ source -> $SLUG@$BR"

## 5. Deploy onto kagent (runtime #1)

Binds the agent to the `kind-kagent` runtime. The registry stands up the agent **and** the referenced MCP server as pods, wired together.

In [ ]:
arctl apply -f yaml/deploy-kagent.yaml

In [ ]:
kubectl --context kind-agentcore-demo -n kagent get pods

## 6. Deploy the same agent to AWS Bedrock AgentCore (runtime #2)

The *same* published agent, now to a second runtime. The AWS Bedrock AgentCore platform was **connected once at setup** (cross-account role + registered runtime), so this just makes the agent multi-cloud, pushes its image and source, and deploys it against the `aws-agentcore` platform — then waits for it to go READY:

In [ ]:
source scripts/aws-login.sh

In [ ]:
./scripts/agentcore-deploy.sh

In [ ]:
./scripts/ac-invoke.sh "Roll a 13-sided die, add 5 to the result, then tell me if it is prime."

## 7. Run it from the kagent UI: traces and spans

Open the **kagent UI** (http://localhost:18007) and invoke `agentdemo` with a prompt that exercises the whole chain (paste it in):

Watch the OTEL trace: the agent calls `roll_die`, then the MCP `sum` tool, then `check_prime`, span by span.

## 8. Govern the tools with an AccessPolicy (least privilege)

The `everything-server` exposes a sensitive `printenv` tool. A platform owner restricts the agent to only the tools it needs. `accesspolicy-on.sh` labels the MCP server for an **agentgateway waypoint** and applies a kagent **AccessPolicy** that denies `printenv`.

In [ ]:
# Least privilege: DENY the agent the `printenv` tool on the everything-server,
# enforced at an agentgateway waypoint. The other tools (sum/echo/...) keep working.
set -o pipefail
CTX=kind-agentcore-demo
MCP=demo-everything-server-agentdemo
AGENT=$(kubectl --context $CTX -n kagent get agents.kagent.dev -o name 2>/dev/null | sed 's#.*/##' | grep -i '^agentdemo' | head -1)
[ -n "$AGENT" ] || { echo "no agentdemo agent deployed on kagent"; exit 1; }
echo "agent=$AGENT  mcpserver=$MCP"

echo "▸ 1/3 route the MCP server through an agentgateway waypoint"
kubectl --context $CTX -n kagent label mcpserver $MCP kagent.solo.io/waypoint=true --overwrite
WP=mcpserver-$MCP-waypoint
for i in $(seq 1 30); do
  kubectl --context $CTX -n kagent get deploy $WP >/dev/null 2>&1 && break; sleep 2
done
kubectl --context $CTX -n kagent rollout status deploy/$WP --timeout=120s || true

echo "▸ 2/3 apply the AccessPolicy (a declarative CR) that denies the printenv tool"
kubectl --context $CTX -n kagent apply -f - <<EOF
apiVersion: policy.kagent-enterprise.solo.io/v1alpha1
kind: AccessPolicy
metadata:
  name: deny-printenv
  namespace: kagent
spec:
  from:
    subjects:
      - kind: Agent
        name: $AGENT
        namespace: kagent
  targetRef:
    kind: MCPServer
    name: $MCP
    tools:
      - printenv
  action: DENY
EOF
echo -n "AccessPolicy state: "; kubectl --context $CTX -n kagent get accesspolicy deny-printenv -o jsonpath='{.status.state}'; echo

echo "▸ 3/3 restart the agent so it re-lists tools through the waypoint"
kubectl --context $CTX -n kagent rollout restart deploy/$AGENT
kubectl --context $CTX -n kagent rollout status deploy/$AGENT --timeout=120s
echo "AccessPolicy enforcing — printenv is now denied"

The policy is a declarative custom resource — show it on the CLI:

In [ ]:
kubectl --context kind-agentcore-demo -n kagent get accesspolicy deny-printenv -o yaml

Ask the agent for its tools again — `printenv` is gone; `sum` and the rest still work. Run it on the CLI (cell below), or paste this prompt into the kagent UI:

In [ ]:
./scripts/ask.sh "List the exact names of every tool you can call. Output only a comma-separated list, nothing else."

## Reset / teardown

Run any of these **in a terminal** (copy each line):